<a href="https://colab.research.google.com/github/sumin031130/ESAA/blob/main/ESAA_YB_WEEK12_1_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [ESAA 수상작 리뷰] DACON 심장 질환 예측 – One-hot + RandomForest (Private 0.92156)

## 1. 데이터

- train **151행 × 15열**, test **152행**
- 변수: `age`, `sex`, `cp`(흉통 유형), `trestbps`(안정시 혈압), `chol`(콜레스테롤), `fbs`(공복 혈당), `restecg`(심전도), `thalach`(최대 심박수), `exang`(운동 유발 협심증), `oldpeak`, `slope`, `ca`, `thal`
- 타깃: `target` (0/1)
- **범주형 8개**: cp, sex, fbs, restecg, exang, slope, ca, thal
- **수치형 5개**: age, trestbps, chol, thalach, oldpeak
- 결측치 없음 (`df.isna().sum()`이 전부 0)

## 2. 코드 흐름

| 단계 | 하는 일 |
|---|---|
| 1 | 라이브러리·한글 폰트 설정, train/test/sample_submission 불러오기 |
| 2 | 범주형 8개를 `pd.get_dummies`로 **원-핫 인코딩** (train·test 각각) |
| 3 | 수치형 5개를 `StandardScaler`로 표준화 |
| 4 | 결측치 확인 |
| 5 | (주석 처리) Stepwise 변수 선택 → R에서 로지스틱으로 고른 변수 8개 → 로지스틱 회귀 |
| 6 | **최종: RandomForest를 전체 변수로 학습** |
| 7 | test에만 있는 `ca_4` 컬럼 제거 → 예측 → 제출 파일 저장 |

## 3. 주요 코드 분석

### 3-1. 범주형 변수 원-핫 인코딩

In [ ]:
dummy_cols = ['cp', 'sex', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
df = pd.get_dummies(df, columns = dummy_cols)

- `cp`, `ca`, `thal`처럼 0,1,2,3으로 적힌 값은 크기가 아니라 **범주 번호**인 경우가 많다. 원-핫으로 바꾸면 모델이 순서를 숫자 크기로 오해하지 않는다. 이 데이터에서 가장 중요한 전처리다.
- 컬럼 수가 train 29개, test 30개로 다르다. test에만 `ca=4`인 사례가 있어 `ca_4`가 생겼기 때문이다.

### 3-2. 불일치하는 컬럼 맞추기

In [ ]:
set(testcol) - set(col)      # {'ca_4'}
test2.drop(['ca_4'], axis = 1, inplace = True)

- train과 test에 따로 `get_dummies`를 하면 이런 불일치가 생긴다. 집합 차(`set(...) - set(...)`)로 찾아 직접 지웠다. 이렇게 하면 `ca_4`인 행은 `ca_0`~`ca_3`이 모두 0인 행이 된다.
- 더 안전한 방법은 train의 컬럼에 맞춰 `test.reindex(columns=..., fill_value=0)`를 쓰는 것이다.

### 3-3. 수치형 표준화

In [ ]:
scaler = StandardScaler()
df[num_col] = scaler.fit_transform(df[num_col])      # train
test[num_col] = scaler.fit_transform(test[num_col])  # test

- train과 test를 **각각의 평균·표준편차로** 표준화했다. 원칙은 train에서 `fit`한 스케일러로 test를 `transform`만 하는 것이다. test를 따로 fit하면 분포가 다를 때 같은 값이 다른 숫자로 바뀐다.
- RandomForest는 트리 모델이라 스케일에 민감하지 않아 **이 표준화는 최종 모델에는 영향이 거의 없다.

### 3-4. (주석 처리된) Stepwise 변수 선택

In [ ]:
# R에서 logistic으로 선택해서 가져옴
# selected_variables = ["chol", "oldpeak", "cp_0", "cp_1", "slope_1", "ca_0", "thal_1", "thal_2"]
# clf = LogisticRegression(random_state=0).fit(X, y)
# clf.score(X, y)   # 0.8874 (train accuracy)

- p-value로 변수를 하나씩 넣고 빼는 **Stepwise 선택**을 구현했고, R에서 로지스틱으로 고른 변수 8개를 써 보았다. 통계학 수업에서 배운 방법을 실제 대회 데이터에 적용한 흔적이다.
- 이 셀들은 **주석 처리**되어 있고 최종 제출에는 쓰이지 않았다. 출력(0.8874)은 이전 실행의 결과이며 **학습 데이터 기준 정확도**라 일반화 성능이 아니다.

### 3-5. 최종 모델: RandomForest

In [ ]:
clf = RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1)
clf.fit(X, y)
pred = clf.predict(test2)

- 트리 10개, 깊이 5, 분기마다 **변수 1개만 후보**(`max_features=1`)로 가볍고 랜덤한 설정이다. 151행의 작은 데이터에서 과적합을 피하려는 설정으로 보인다.
- 제출 파일명에 `ALL`이 붙어 있어 **변수 선택 없이 전체 변수**를 쓴 제출임을 알 수 있다.

## 4. 새롭게 알게 된 점

- **데이터가 작으면 단순한 전처리와 가벼운 모델이 충분할 수 있다.** 151행에서는 학습할 정보 자체가 적다.
- **숫자로 적힌 범주형 변수**(cp, ca, thal 등)를 구분해서 원-핫으로 인코딩하는 일이 중요하다.
- train과 test를 따로 `get_dummies`하면 컬럼이 달라질 수 있고, `set` 차이로 찾아 맞추는 방법이 있다.


## 5. 아쉬운 점

- **검증 과정이 없다.** `train_test_split`을 import했지만 쓰지 않았고, 교차검증도 없다. 최종 모델의 정확도를 노트북 안에서 한 번도 확인하지 않는다. Private 0.92156이 나왔지만 이 코드만으로는 왜 좋았는지 설명할 근거가 없다.
- **test를 따로 표준화**한 것은 정석이 아니다.
- **하이퍼파라미터 튜닝이 없고** 다른 모델(XGBoost, LightGBM 등)과의 비교도 없다. `n_estimators=10`, `max_features=1`이 어떤 근거로 정해졌는지 알 수 없다.
- 모델 선택 근거가 부족하다.